# Baseline v7 — Multilingual Cross-Encoder (`mmarco-mMiniLMv2-L12`)

**Vấn đề v5/v6:** CE `ms-marco-MiniLM-L-6-v2` (English-only) không phân biệt được các passages pháp lý tiếng Việt

**Fix v7:** Dùng `cross-encoder/mmarco-mMiniLMv2-L12-H384` — trained trên **26 ngôn ngữ** (có tiếng Việt)

```
v5: v4_FT_Bi + mMiniLM-L6 (EN) CE  → R@1=0.5418, MRR=0.6307
v7: v4_FT_Bi + mMiniLMv2-L12 (Multi) CE →  ?
```

| Model | Params | Languages | R@1 baseline |
|-------|--------|-----------|------|
| ms-marco-MiniLM-L-6-v2 | 22M | **EN only** | 0.5418 |
| **mmarco-mMiniLMv2-L12-H384** | **117M** | **26 langs** | ? |

## Cell 0 — Config

In [1]:
import json, csv, time, random, gc
import numpy as np, faiss, torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

ROOT, DATA_DIR = Path("."), Path(".") / "data"
EVAL_DIR = ROOT / "outputs" / "eval"
TMP_DIR  = ROOT / "outputs" / "tmp"
MDL_DIR  = ROOT / "outputs" / "models"

TRAIN_NEG    = DATA_DIR / "train_with_neg.jsonl"
DEV_FILE     = DATA_DIR / "dev.jsonl"
EVAL_QA_FILE = EVAL_DIR / "eval_qa.jsonl"
FT_BI_PATH   = MDL_DIR  / "legal_hf_finetuned" / "final"
FAISS_V4     = TMP_DIR  / "faiss_v4.index"
MAP_V4       = TMP_DIR  / "faiss_mapping_v4.jsonl"
RESULT_CSV   = EVAL_DIR / "rerank_metrics_v7.csv"
CE_OUT_DIR   = MDL_DIR  / "cross_encoder_v7_multilingual"

# ── KEY CONFIG ──
# Option A: Fine-tune multilingual CE (khuyến nghị)
BASE_CE_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384"
# Option B: Zero-shot (không fine-tune) — uncomment để thử nhanh
# ZERO_SHOT = True
ZERO_SHOT    = False   # True = dùng model gốc không fine-tune

# D_skip14 config (best từ grid search)
SKIP_TOP_K   = 14
TOP_MINE     = 30
HARD_NEG_PER = 2
CE_EPOCHS    = 3      # 3 epochs vì multilingual model lớn hơn
CE_BATCH     = 16     # batch nhỏ hơn do model 117M params
CE_MAX_LEN   = 256
TOP_N_EVAL   = 50
SEED         = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
CE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device      : {DEVICE}")
print(f"CE Model    : {BASE_CE_MODEL}")
print(f"Zero-shot   : {ZERO_SHOT}")
print(f"SKIP_TOP_K  : {SKIP_TOP_K} (range rank {SKIP_TOP_K+1}-{TOP_MINE})")
print(f"CE_BATCH    : {CE_BATCH} (reduced for larger model)")
print(f"Target      : R@1>0.5418, R@5>0.7245, MRR>0.6307 (v5 best)")

d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device      : cuda
CE Model    : cross-encoder/mmarco-mMiniLMv2-L12-H384
Zero-shot   : False
SKIP_TOP_K  : 14 (range rank 15-30)
CE_BATCH    : 16 (reduced for larger model)
Target      : R@1>0.5418, R@5>0.7245, MRR>0.6307 (v5 best)


## Cell 1 — Utilities

In [2]:
def load_jsonl(path):
    rows, err = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: err+=1
    if err: print(f"  ⚠ {err} errors")
    return rows

def is_hit(fid, ec, mapping):
    row=mapping[fid]
    for e in ec:
        ci=e.get("chunk_index",-2)
        if ci!=-1 and row["chunk_index"]==ci: return True
        if row["van_ban"]==e.get("van_ban","") and row["dieu"]==e.get("dieu","") and row["khoan"]==e.get("khoan",""): return True
    return False

def is_pos_meta(cand, meta):
    if cand["van_ban"]==meta.get("van_ban","") and cand["van_ban"]!="" \
       and cand["dieu"]==meta.get("dieu","") and cand["khoan"]==meta.get("khoan",""): return True
    ci=meta.get("chunk_index",-2)
    return ci!=-1 and cand["chunk_index"]==ci

def avg(lst): return round(sum(lst)/len(lst),4) if lst else 0.0
print("Utilities ✓")

Utilities ✓


## Cell 2 — Option B: Zero-shot eval (chạy trước để xem hiệu quả gốc)

In [3]:
# Load bi-encoder + FAISS
print("Loading v4 bi-encoder + FAISS...")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
eval_qa    = load_jsonl(EVAL_QA_FILE)
print(f"Loaded ✓ | {len(eval_qa)} questions")

# Load multilingual CE zero-shot (chưa fine-tune)
print(f"Loading multilingual CE (zero-shot): {BASE_CE_MODEL}")
ce_zeroshot = CrossEncoder(BASE_CE_MODEL, max_length=CE_MAX_LEN, device=DEVICE)
print("CE loaded ✓")

r_base = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}
r_zs   = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}

for item in tqdm(eval_qa, desc="Zero-shot eval"):
    query=item["query"]; ec=item["expected_citations"]
    q_emb=ft_bi.encode([query],normalize_embeddings=True,convert_to_numpy=True).astype("float32")
    _,ids=index_v4.search(q_emb,TOP_N_EVAL)
    ids=ids[0].tolist()

    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_base[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in ids[:k] if i>=0) else 0)
    mrr=0.0
    for rank,i in enumerate(ids[:10],1):
        if i>=0 and is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_base["MRR@10"].append(mrr)

    cands  = [(mapping_v4[i]["passage"],i) for i in ids if i>=0]
    rscore = ce_zeroshot.predict([[query,c[0]] for c in cands],batch_size=CE_BATCH) if cands else []
    ranked = sorted(zip(rscore,[c[1] for c in cands]),reverse=True)
    r_ids  = [x[1] for x in ranked]

    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_zs[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in r_ids[:k]) else 0)
    mrr=0.0
    for rank,i in enumerate(r_ids[:10],1):
        if is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_zs["MRR@10"].append(mrr)

V5 = {"R@1":0.5418,"R@3":0.6873,"R@5":0.7245,"MRR@10":0.6307}
print("\n── Zero-shot Multilingual CE ──")
print(f"  {'Metric':<10} {'v4 Baseline':>14} {'Zero-shot CE':>14} {'v5 (EN CE)':>14}")
print("  "+"-"*54)
for m in ["R@1","R@3","R@5","MRR@10"]:
    b=avg(r_base[m]); zs=avg(r_zs[m])
    win="✅" if zs>V5[m]+0.001 else ("⚠️" if zs>avg(r_base[m]) else "❌")
    print(f"  {m:<10} {b:>14.4f} {zs:>14.4f} {V5[m]:>14.4f} {win}")
print("\n→ Nếu zero-shot đã tốt hơn v5, SKIP Cell 3-4 và dùng ce_zeroshot luôn!")
print("→ Nếu chưa tốt, chạy tiếp Cell 3-4 để fine-tune.")

Loading v4 bi-encoder + FAISS...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 432.83it/s, Materializing param=pooler.dense.weight]                               


Loaded ✓ | 323 questions
Loading multilingual CE (zero-shot): cross-encoder/mmarco-mMiniLMv2-L12-H384


OSError: cross-encoder/mmarco-mMiniLMv2-L12-H384 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

## Cell 3 — Mine Hard Negatives (D_skip14)
> Bỏ qua nếu zero-shot đã đủ tốt

In [ ]:
pos_rows = [r for r in load_jsonl(TRAIN_NEG) if r.get("label")==1]
random.seed(SEED); random.shuffle(pos_rows)
print(f"Mining HN D_skip14 (rank {SKIP_TOP_K+1}-{TOP_MINE})...")

train_data = []
stats={"pos":0,"neg":0,"no_neg":0}
for r in tqdm(pos_rows, desc="Mining"):
    query=r.get("query","").strip(); pos_p=r.get("passage","").strip(); meta=r.get("meta",{})
    if not query or not pos_p: continue
    train_data.append(InputExample(texts=[query,pos_p],label=1.0)); stats["pos"]+=1
    q_emb=ft_bi.encode([query],normalize_embeddings=True,convert_to_numpy=True).astype("float32")
    _,ids=index_v4.search(q_emb,TOP_MINE); ids=ids[0].tolist()
    added=0
    for fid in ids[SKIP_TOP_K:]:
        if fid<0 or added>=HARD_NEG_PER: break
        cand=mapping_v4[fid]
        if cand["passage"]==pos_p or is_pos_meta(cand,meta): continue
        train_data.append(InputExample(texts=[query,cand["passage"]],label=0.0))
        added+=1; stats["neg"]+=1
    if added==0: stats["no_neg"]+=1

# Dev
dev_rows=load_jsonl(DEV_FILE); random.seed(SEED); random.shuffle(dev_rows)
dev_data=[]
for r in tqdm(dev_rows[:500],desc="Dev",leave=False):
    query=r.get("query","").strip(); pos_p=r.get("passage","").strip(); meta=r.get("meta",{})
    if not query or not pos_p: continue
    dev_data.append(InputExample(texts=[query,pos_p],label=1.0))
    q_emb=ft_bi.encode([query],normalize_embeddings=True,convert_to_numpy=True).astype("float32")
    _,ids=index_v4.search(q_emb,TOP_MINE); ids=ids[0].tolist()
    for fid in ids[SKIP_TOP_K:]:
        if fid<0: break
        cand=mapping_v4[fid]
        if cand["passage"]==pos_p or is_pos_meta(cand,meta): continue
        dev_data.append(InputExample(texts=[query,cand["passage"]],label=0.0)); break

print(f"Train: {len(train_data)} | Dev: {len(dev_data)} | neg/q={stats['neg']/max(stats['pos'],1):.1f}")

## Cell 4 — Fine-tune Multilingual CE
> ⏱️ ~30-45 phút (model lớn hơn 5× so với v5 CE)

In [ ]:
# Giải phóng bi-encoder VRAM
del ft_bi; gc.collect()
torch.cuda.empty_cache() if DEVICE=="cuda" else None
print("VRAM cleared ✓")

# Dùng lại ce_zeroshot (đã load ở Cell 2) hoặc load lại
ce_ft = CrossEncoder(BASE_CE_MODEL, num_labels=1, max_length=CE_MAX_LEN, device=DEVICE)
evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_data, name="v7")
warmup=int(len(train_data)/CE_BATCH * CE_EPOCHS * 0.1)
random.seed(SEED); random.shuffle(train_data)

print(f"Train: {len(train_data)} | Epochs: {CE_EPOCHS} | Batch: {CE_BATCH} | Warmup: {warmup}")
t0=time.time()
ce_ft.fit(
    train_dataloader=DataLoader(train_data,shuffle=True,batch_size=CE_BATCH),
    evaluator=evaluator, epochs=CE_EPOCHS,
    warmup_steps=warmup, output_path=str(CE_OUT_DIR),
    use_amp=(DEVICE=="cuda"),
)
elapsed=round((time.time()-t0)/60,1)
print(f"Training done in {elapsed} min")
saved=CE_OUT_DIR/"saved_model"; ce_ft.save(str(saved))
print(f"Saved → {saved}")

## Cell 5 — Evaluate v7 Fine-tuned & So sánh toàn bộ

In [ ]:
gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None
ft_bi=SentenceTransformer(str(FT_BI_PATH),device=DEVICE)
index_v4=faiss.read_index(str(FAISS_V4)); mapping_v4=load_jsonl(MAP_V4)
eval_qa=load_jsonl(EVAL_QA_FILE)
print(f"Loaded ✓ | {len(eval_qa)} questions")

r_ft={"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}

for item in tqdm(eval_qa,desc="Eval v7 fine-tuned"):
    query=item["query"]; ec=item["expected_citations"]
    q_emb=ft_bi.encode([query],normalize_embeddings=True,convert_to_numpy=True).astype("float32")
    _,ids=index_v4.search(q_emb,TOP_N_EVAL); ids=ids[0].tolist()
    cands=[(mapping_v4[i]["passage"],i) for i in ids if i>=0]
    rscore=ce_ft.predict([[query,c[0]] for c in cands],batch_size=CE_BATCH) if cands else []
    ranked=sorted(zip(rscore,[c[1] for c in cands]),reverse=True)
    r_ids=[x[1] for x in ranked]
    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_ft[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in r_ids[:k]) else 0)
    mrr=0.0
    for rank,i in enumerate(r_ids[:10],1):
        if is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_ft["MRR@10"].append(mrr)

V7  = {m:avg(r_ft[m]) for m in ["R@1","R@3","R@5","MRR@10"]}
V7z = {m:avg(r_zs[m]) for m in ["R@1","R@3","R@5","MRR@10"]}  # zero-shot từ Cell 2

print("\n" + "="*90)
print(f"  {'Metric':<10} {'v5(EN_CE)':>12} {'v7_ZeroShot':>13} {'v7_FineTune':>13} {'Δ(v7FT-v5)':>12}")
print("="*90)
for m in ["R@1","R@3","R@5","MRR@10"]:
    d=V7[m]-V5[m]; sign="+" if d>=0 else ""
    win="✅" if d>0.001 else ("❌" if d<-0.001 else "=")
    print(f"  {m:<10} {V5[m]:>12.4f} {V7z[m]:>13.4f} {V7[m]:>13.4f} {sign}{d:>11.4f} {win}")
print("="*90)

rows=[{"metric":m,"v5_EN_ce":V5[m],"v7_zeroshot":V7z[m],"v7_finetuned":V7[m]} for m in ["R@1","R@3","R@5","MRR@10"]]
with open(RESULT_CSV,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["metric","v5_EN_ce","v7_zeroshot","v7_finetuned"])
    w.writeheader(); w.writerows(rows)
print(f"Saved → {RESULT_CSV} ✓")